## Strokes Gained Calc

In [1]:
%pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np

In [4]:
es_df = pd.read_csv('/Users/masonscott/git-repos/Python-Projects/golf_app/data/strokes/expected_strokes.csv')
esg_df = pd.read_csv('/Users/masonscott/git-repos/Python-Projects/golf_app/data/strokes/expected_strokes_(green).csv')

In [5]:
expected_strokes = {}

skills = ['PGA Tour', 'Scratch', '5 handicap', '10 handicap', '15 handicap', '20 handicap', '25 handicap']

for skill in skills:
  for lie in set(es_df['Lie'].tolist()):
    expected_strokes[lie, skill] = es_df[es_df['Lie'] == lie]['Yards'].tolist(), es_df[es_df['Lie'] == lie][skill].tolist()
  expected_strokes['Green', skill] = esg_df['Feet'].tolist(), esg_df[skill].tolist()

In [6]:
def get_expected_strokes(lie: str, distance: int, skill: str):
    distances, es = expected_strokes[(lie, skill)]
    if lie == 'Green':
      distance = np.log(distance) if distance >= 1 else np.log(1)
      distances = np.log(distances)
    return float(np.interp(distance, distances, es))

In [7]:
def get_strokes_gained(lie1: str, lie2: str, dist1: int, dist2: int, skill: str, penalty_strokes: int = 0):
  if dist2 == 0 and lie2 == 'Holed':
    return get_expected_strokes(lie1, dist1, skill) - 1 - penalty_strokes
  return get_expected_strokes(lie1, dist1, skill) - get_expected_strokes(lie2, dist2, skill) - 1 - penalty_strokes

In [8]:
def print_strokes_gained(lie1, lie2, distance1, distance2, skill, penalty_strokes=0, description: str = ""):
  print(f'Starting Lie: {lie1}\nStarting Distance: {distance1}\nEnding Lie: {lie2}\nEnding Distance: {distance2}\nSkill: {skill}\nPenalty Strokes: {penalty_strokes}\nDescription: {description}\nStrokes Gained: {get_strokes_gained(lie1, lie2, distance1, distance2, skill, penalty_strokes):+.2f}\n\n')

In [9]:
def print_strokes_gained_chain(hole_dict: dict) -> None:
  skill = hole_dict['skill']
  starting_pos = hole_dict['tee']
  strokes_lst = hole_dict['strokes']

  total_strokes_gained = 0.0
  for stroke in strokes_lst:
    lie1, dist1 = starting_pos['lie'], starting_pos['distance']
    lie2, dist2, penalty_strokes, description = stroke['lie'], stroke['distance'], stroke['penalty_strokes'], stroke['description']

    sg = get_strokes_gained(lie1=lie1, lie2=lie2, dist1=dist1, dist2=dist2, skill=skill, penalty_strokes=penalty_strokes)
    print_strokes_gained(lie1, lie2, dist1, dist2, skill, penalty_strokes, description=description)

    total_strokes_gained += sg

    starting_pos = stroke

  lie1, dist1 = starting_pos['lie'], starting_pos['distance']
  sg = get_strokes_gained(lie1=lie1, lie2='Holed', dist1=dist1, dist2=0, skill=skill, penalty_strokes=0)
  print_strokes_gained(lie1, 'Holed', dist1, 0, skill, 0, description="Holed out")

  total_strokes_gained += sg
  print(f'Total Strokes Gained: {total_strokes_gained:+.2f}')

In [10]:
hole_1 = {
    'skill': '15 handicap',
    'tee': {
      'distance': 353,
      'lie': "Tee",
      'penalty_strokes': 0,
    },
    'strokes': [
        {
          'distance': 75,
          'lie': 'Rough',
          'penalty_strokes': 0,
          'description': "Hooked 5 iron off the tee",
        },
        {
          'distance': 60,
          'lie': 'Rough',
          'penalty_strokes': 1,
          'description': "Topped approach into hazard",
        },
        {
          'distance': 20,
          'lie': 'Green',
          'penalty_strokes': 0,
          'description': "Got onto the green",
        },
        {
          'distance': 5,
          'lie': 'Green',
          'penalty_strokes': 0,
          'description': "Hammered putt passed the hole",
        },
        {
          'distance': 1,
          'lie': 'Green',
          'penalty_strokes': 0,
          'description': "Left putt short",
        },
    ],
 }

In [11]:
print_strokes_gained_chain(hole_1)

Starting Lie: Tee
Starting Distance: 353
Ending Lie: Rough
Ending Distance: 75
Skill: 15 handicap
Penalty Strokes: 0
Description: Hooked 5 iron off the tee
Strokes Gained: +0.23


Starting Lie: Rough
Starting Distance: 75
Ending Lie: Rough
Ending Distance: 60
Skill: 15 handicap
Penalty Strokes: 1
Description: Topped approach into hazard
Strokes Gained: -1.96


Starting Lie: Rough
Starting Distance: 60
Ending Lie: Green
Ending Distance: 20
Skill: 15 handicap
Penalty Strokes: 0
Description: Got onto the green
Strokes Gained: +0.41


Starting Lie: Green
Starting Distance: 20
Ending Lie: Green
Ending Distance: 5
Skill: 15 handicap
Penalty Strokes: 0
Description: Hammered putt passed the hole
Strokes Gained: -0.38


Starting Lie: Green
Starting Distance: 5
Ending Lie: Green
Ending Distance: 1
Skill: 15 handicap
Penalty Strokes: 0
Description: Left putt short
Strokes Gained: -0.74


Starting Lie: Green
Starting Distance: 1
Ending Lie: Holed
Ending Distance: 0
Skill: 15 handicap
Penalty Strok

## Golf Course API

In [14]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.


In [16]:
import json

path = '/Users/masonscott/git-repos/Python-Projects/golf_app/data/courses/holly_hills.json'
with open(path, 'r') as file:
    course = json.load(file)['course']

In [17]:
print(course.keys())

dict_keys(['id', 'club_name', 'course_name', 'location', 'tees'])


In [18]:
mens_tees = course['tees']['male']
white_tees = next((tees for tees in mens_tees if tees.get('tee_name') == 'White'), None)

In [19]:
white_tees

rating = white_tees['course_rating']
slope = white_tees['slope_rating']
par = white_tees['par_total']
holes = white_tees['holes']

In [20]:
holes

[{'par': 4, 'yardage': 353, 'handicap': 11},
 {'par': 4, 'yardage': 343, 'handicap': 5},
 {'par': 3, 'yardage': 149, 'handicap': 15},
 {'par': 4, 'yardage': 372, 'handicap': 1},
 {'par': 5, 'yardage': 517, 'handicap': 13},
 {'par': 3, 'yardage': 173, 'handicap': 9},
 {'par': 4, 'yardage': 290, 'handicap': 17},
 {'par': 4, 'yardage': 386, 'handicap': 3},
 {'par': 5, 'yardage': 519, 'handicap': 7},
 {'par': 4, 'yardage': 422, 'handicap': 4},
 {'par': 4, 'yardage': 391, 'handicap': 10},
 {'par': 4, 'yardage': 373, 'handicap': 8},
 {'par': 5, 'yardage': 492, 'handicap': 16},
 {'par': 3, 'yardage': 139, 'handicap': 18},
 {'par': 4, 'yardage': 444, 'handicap': 2},
 {'par': 3, 'yardage': 179, 'handicap': 12},
 {'par': 4, 'yardage': 391, 'handicap': 6},
 {'par': 5, 'yardage': 497, 'handicap': 14}]